# حاسوب زهراء الكروي: التحديث الأسبوعي (النسخة 2)

1. Runtime ← Run all.
2. لما يطلب، ارفعي ملف المشروع `zahraa-supercomputer.zip`.
3. بالنهاية ينزل `zahraa_web.zip`: الصفحة والبيانات الجديدة، جاهزة للنشر (ابعثيها لـClaude أو ارفعيها لأي استضافة).

التحديث آمن: إذا مصدر عطّل، تبقى آخر بيانات سليمة؛ وإذا فشل فحص البيانات، ما يتغير شي.

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()
name = next(iter(up))
zipfile.ZipFile(name).extractall('.')
os.chdir('zahraa-supercomputer')
print('المشروع جاهز:', os.getcwd())

In [ ]:
!pip -q install numpy==2.5.3 pandas==3.0.5 scipy==1.18.1
import numpy, pandas, scipy
print('numpy', numpy.__version__, '· pandas', pandas.__version__, '· scipy', scipy.__version__)

In [ ]:
!python -m zsc update

In [ ]:
import json, pandas as pd
meta = json.load(open('app/data/meta.json'))
print('آخر تحديث:', meta['generated_utc'], '· البيانات', meta['versions']['data'])
fx = pd.DataFrame(json.load(open('app/data/fixtures.json'))['fixtures'])
teams = json.load(open('app/data/teams.json'))['teams']
up = fx[fx['score'].isna() & fx['p'].notna()].sort_values('kickoff_utc').head(10).copy()
up['المباراة'] = up.apply(lambda r: f"{teams[r.home]['ar']} × {teams[r.away]['ar']}", axis=1)
up[['فوز الأرض', 'تعادل', 'فوز الضيف']] = pd.DataFrame(up['p'].tolist(), index=up.index).mul(100).round(0)
display(up[['المباراة', 'kickoff_utc', 'فوز الأرض', 'تعادل', 'فوز الضيف']])

In [ ]:
s = json.load(open('app/data/season.json'))
t = pd.DataFrame(s['projection'])
t['الفريق'] = t['id'].map(lambda i: teams[i]['ar'])
cols = {'exp_pts': 'النقاط المتوقعة', 'p_title': 'اللقب', 'p_ucl': 'الأبطال', 'p_releg': 'الهبوط'}
t = t.sort_values('exp_pts', ascending=False)[['الفريق'] + list(cols)].rename(columns=cols)
for c in ['اللقب', 'الأبطال', 'الهبوط']: t[c] = (100 * t[c]).round(1)
display(t.round(1))

In [ ]:
!python scripts/build_app.py
import shutil
shutil.make_archive('zahraa_web', 'zip', 'dist/web')
files.download('zahraa_web.zip')